In [5]:
import numpy as np
from Bio.PDB import PDBParser, Superimposer, PDBList
import os

def determine_structural_similarity(pdb_id_ref, pdb_id_sample):
    # 1. Setup Parser and Downloader
    parser = PDBParser(QUIET=True)
    pdbl = PDBList()
    
    # Download files if they don't exist locally
    for p_id in [pdb_id_ref, pdb_id_sample]:
        if not os.path.exists(f'pdb{p_id.lower()}.ent'):
            pdbl.retrieve_pdb_file(p_id, pdir='.', file_format='pdb')

    # 2. Load the structures
    # Using the standard filename format provided by PDBList
    ref_struct = parser.get_structure("reference", f"pdb{pdb_id_ref.lower()}.ent")
    sample_struct = parser.get_structure("sample", f"pdb{pdb_id_sample.lower()}.ent")
    
    # 3. Extract Alpha-Carbon (CA) atoms
    # Use a dictionary or list to ensure the comparison matching only residues
    ref_atoms = []
    sample_atoms = []
    
    # Iterate through the first model of each structure
    ref_model = ref_struct[0]
    sample_model = sample_struct[0]

    for chain in ref_model:
        for res in chain:
            if 'CA' in res and res.get_id()[0] == ' ': # Ensure it's a standard residue
                ref_atoms.append(res['CA'])
                    
    for chain in sample_model:
        for res in chain:
            if 'CA' in res and res.get_id()[0] == ' ':
                sample_atoms.append(res['CA'])

    # 4. Critical Step: Align Atom Counts
    # RMSD calculation requires exactly the same number of atoms
    length = min(len(ref_atoms), len(sample_atoms))
    ref_atoms = ref_atoms[:length]
    sample_atoms = sample_atoms[:length]

    # 5. Superimpose (Align) the structures
    super_imposer = Superimposer()
    super_imposer.set_atoms(ref_atoms, sample_atoms)
    
    # This physically shifts the sample coordinates to match the reference
    super_imposer.apply(sample_atoms)
    
    # 6. Output Analysis
    print(f"--- Structural Determination Analysis ---")
    print(f"Reference: {pdb_id_ref} | Sample: {pdb_id_sample}")
    print(f"Number of atoms aligned: {length}")
    print(f"RMSD Result: {super_imposer.rms:.4f} Å")
    
    if super_imposer.rms < 2.0:
        print("Conclusion: High structural similarity (Match).")
    elif super_imposer.rms < 5.0:
        print("Conclusion: Moderate similarity. Possible conformational shift.")
    else:
        print("Conclusion: Significant structural divergence.")

# Execute the analysis, comparing CRBN (4OO2) to the complexed version (6H0F)
determine_structural_similarity('4OO2', '6H0F')

--- Structural Determination Analysis ---
Reference: 4OO2 | Sample: 6H0F
Number of atoms aligned: 1979
RMSD Result: 54.7196 Å
Conclusion: Significant structural divergence.
